In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import f1_score, accuracy_score

from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D
from tensorflow.keras.layers import GlobalMaxPooling1D, GlobalAveragePooling1D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
VOCAB_SIZE = 15000
MAX_LENGTH = 50
EMBEDDING_DIM = 32

LEARNING_RATE = 5e-4
BATCH_SIZE = 64
EPOCHS = 10

RANDOM_STATE = 42

In [ ]:
def clean_text(text):
    # remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # remove @mentions
    text = re.sub(r'@\w+', '', text)
    ## remove #topics
    #text = re.sub(r'#\w+', '', text)
    # remove non-alphabetical characters but keep white spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # lower case text
    text = text.lower()
    return ' '.join(text.split())

In [ ]:
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

df

In [ ]:
df['text_clean'] = df['text'].apply(clean_text)
df_test['text_clean'] = df_test['text'].apply(clean_text)

In [ ]:
text, text_clean = df[df["text"].str.contains("#")].sample()[["text", "text_clean"]].values[0]
print("Raw text:\n", text)
print("Clean text:\n", text_clean)

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text_clean"])

X = tokenizer.texts_to_sequences(df["text_clean"])
X = pad_sequences(X, maxlen=MAX_LENGTH, padding='post')

X_test = tokenizer.texts_to_sequences(df_test["text_clean"])
X_test = pad_sequences(X_test, maxlen=MAX_LENGTH, padding='post')

y = df["target"].values

print(X.shape)
print(X_test.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, 
                                                  test_size=0.2, 
                                                  stratify=y, 
                                                  random_state=RANDOM_STATE)

In [ ]:
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM),
    Conv1D(32, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
], name="ConvEmbedding")

model.build(input_shape=(BATCH_SIZE, MAX_LENGTH))
model.summary()

In [ ]:
model.compile(loss="binary_crossentropy", 
              optimizer=Adam(LEARNING_RATE), 
              metrics=["accuracy"])

In [ ]:
#model.save_weights("initial_model.weights.h5")

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', save_best_only=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=1e-4)

In [ ]:
fitting = model.fit(X_train, y_train, 
                    validation_data=(X_val, y_val),  
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS, 
                    callbacks=[early_stopping, model_checkpoint, reduce_lr])

model.load_weights("best_model.keras")

In [ ]:
epoch_counts = range(1, early_stopping.stopped_epoch+2)
plt.plot(epoch_counts, fitting.history["loss"])
plt.plot(epoch_counts, fitting.history["val_loss"])
plt.legend(["Training", "Validation"])
plt.title("Loss")
plt.show()

In [ ]:
plt.plot(epoch_counts, fitting.history["accuracy"])
plt.plot(epoch_counts, fitting.history["val_accuracy"])
plt.legend(["Training", "Validation"])
plt.title("Accuracy")
plt.show()

In [ ]:
y_proba = model.predict(X_val)
y_pred = (y_proba >= 0.5).astype(int)

print(y_proba)
print(y_pred)

In [ ]:
print(classification_report(y_val, y_pred))

In [ ]:
cm = confusion_matrix(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average='weighted')

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Disaster', 'Disaster'],
            yticklabels=['Not Disaster', 'Disaster'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix\nF1 Score (weighted): {f1:.3f}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#model.load_weights("initial_model.weights.h5")

In [ ]:
#model.fit(X, y, batch_size=BATCH_SIZE, epochs=EPOCHS)

In [ ]:
y_proba = model.predict(X_test)
y_pred = (y_proba >= 0.5).astype(int)

df_test["prediction"] = y_pred

df_test.head()

In [ ]:
df_submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
df_submission["target"] = y_pred

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)